## 0. Setup & Config

Setup dependencies, parameters, folders, logging, styles and portfolio assumptions.

| Cell | What it does | Output |
|---|---|---|
| Main | Executes this section according to the SWS Portfolio Analysis Model standard | Saved tables/figures/dashboard artifacts |

$$w_i=V_i/\sum_j V_j$$

The portfolio model is an aggregation layer: company-level analysis becomes decision-useful only after weighting by real holdings and cash flows.


In [ ]:
# 0.1 Silent install, imports, config, outputs, logging and style
import subprocess, sys
required_packages = ["pandas", "numpy", "matplotlib", "seaborn", "plotly", "openpyxl", "xlsxwriter"]
for package in required_packages:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
    except Exception as exc:
        print(f"[WARNING] Failed to install {package}: {exc}")

import os, json, logging, importlib.util
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from portfolio_analysis.src import PortfolioConfig, analyze_portfolio, export_portfolio_analysis, make_equal_weight_watchlist

np.random.seed(42)
EXPERIMENT = {
    "name": "sws_portfolio_analysis_model",
    "start_date": "2021-01-01",
    "end_date": "2026-01-01",
    "target": "portfolio_total_return",
    "horizons": [21, 63, 252],
    "test_start": "2024-01-01",
    "embargo": 21,
    "n_quantiles": 5,
    "cost_bps": 10.0,
    "models": ["portfolio_attribution"],
    "run_ablation": True,
    "run_backtest": True,
    "save_figures": True,
    "feature_blocks": ["value", "future", "past", "health", "income", "management", "allocation"],
    "base_currency": "EUR",
    "portfolio_value": 100000.0,
    "synthetic_seed": 42,
}
PORTFOLIO = {"name": "Demo SWS Portfolio", "base_currency": EXPERIMENT["base_currency"], "benchmark": "SPY"}
OUTPUT_ROOT = Path("output")
FIGURES_DIR, TABLES_DIR, LOGS_DIR, DASHBOARD_DIR = [OUTPUT_ROOT / x for x in ["figures", "tables", "logs", "dashboard"]]
for d in [OUTPUT_ROOT, FIGURES_DIR, TABLES_DIR, LOGS_DIR, DASHBOARD_DIR]:
    d.mkdir(parents=True, exist_ok=True)
logger = logging.getLogger(EXPERIMENT["name"])
logger.setLevel(logging.INFO)
logger.handlers = []
fmt = logging.Formatter("%(asctime)s - %(levelname)s - %(message)s")
fh = logging.FileHandler(LOGS_DIR / "experiment.log"); fh.setFormatter(fmt)
sh = logging.StreamHandler(); sh.setFormatter(fmt)
logger.addHandler(fh); logger.addHandler(sh)
COLORS = {"primary":"#01696f","accent":"#da7101","q1":"#c0392b","neutral":"#7a7974","bg":"#f7f6f2","blue":"#006494","gold":"#d19900","purple":"#7a39bb"}
MODEL_COLORS = {"portfolio": COLORS["primary"], "benchmark": COLORS["accent"], "active": COLORS["purple"]}
plt.rcParams["figure.dpi"] = 180
plt.rcParams["figure.figsize"] = (10,5)
plt.rcParams["axes.facecolor"] = COLORS["bg"]
plt.rcParams["savefig.facecolor"] = COLORS["bg"]
sns.set_theme(style="whitegrid", font_scale=1.05)
PLOTLY_AVAILABLE = importlib.util.find_spec("plotly") is not None
if PLOTLY_AVAILABLE:
    import plotly.express as px
    import plotly.graph_objects as go
logger.info("Portfolio analysis initialized | Plotly=%s", PLOTLY_AVAILABLE)


## 1. Data Ingestion

Load transactions, latest prices, company scores and optional benchmark. Real files are attempted first; realistic demo data is generated if absent.

| Cell | What it does | Output |
|---|---|---|
| Main | Executes this section according to the SWS Portfolio Analysis Model standard | Saved tables/figures/dashboard artifacts |

$$V_i=q_i P_i FX_i$$

SWS distinguishes watchlists, holdings and transaction portfolios. This notebook supports all via a transaction ledger or generated pseudo-holdings.


In [ ]:
# 1.1 Load transactions, company scores, latest prices and price history
rng = np.random.default_rng(EXPERIMENT["synthetic_seed"])
input_dir = Path("data")
transactions_path = input_dir / "portfolio_transactions.csv"
company_scores_path = Path("company_valuation/output/tables/Table_XII_full_sws_company_analysis_scores.csv")
company_ranking_path = Path("company_valuation/output/tables/Table_V_company_ranking.csv")

try:
    transactions = pd.read_csv(transactions_path)
    source_transactions = "local_csv"
except Exception as exc:
    logger.warning("Transactions file unavailable; creating realistic demo ledger: %s", exc)
    tickers = ["AAPL", "MSFT", "ASML.AS", "UCG.MI", "ISP.MI", "SAN.MC", "BNP.PA", "SAP.DE"]
    rows = []
    for i, ticker in enumerate(tickers):
        rows.append({"ticker": ticker, "date": "2021-01-04", "transaction_type": "buy", "shares": 8 + i * 3, "price": 80 + i * 12, "fx_rate": 1.0, "fees": 2.0})
        if i % 3 == 0:
            rows.append({"ticker": ticker, "date": "2023-06-15", "transaction_type": "buy", "shares": 2 + i, "price": 100 + i * 14, "fx_rate": 1.0, "fees": 1.0})
    transactions = pd.DataFrame(rows)
    transactions["transactions_synthetic"] = True
    source_transactions = "synthetic_demo_ledger"

try:
    if company_scores_path.exists():
        company_scores = pd.read_csv(company_scores_path)
    elif company_ranking_path.exists():
        company_scores = pd.read_csv(company_ranking_path)
    else:
        raise FileNotFoundError("company valuation score files unavailable")
    source_company_scores = "company_valuation_outputs"
except Exception as exc:
    logger.warning("Company scores unavailable; creating synthetic SWS-style company scores: %s", exc)
    tickers = sorted(transactions["ticker"].str.upper().unique())
    company_scores = pd.DataFrame({
        "ticker": tickers,
        "sws_value_score": rng.integers(1, 7, len(tickers)),
        "sws_future_score": rng.integers(1, 7, len(tickers)),
        "sws_past_score": rng.integers(1, 7, len(tickers)),
        "sws_health_score": rng.integers(1, 7, len(tickers)),
        "sws_income_score": rng.integers(1, 7, len(tickers)),
        "fair_value": rng.uniform(80, 220, len(tickers)),
        "pe_ratio": rng.uniform(8, 35, len(tickers)),
        "pb_ratio": rng.uniform(0.7, 6, len(tickers)),
        "beta": rng.uniform(0.7, 1.6, len(tickers)),
        "future_earnings_growth": rng.uniform(-0.03, 0.18, len(tickers)),
        "future_revenue_growth": rng.uniform(0.00, 0.14, len(tickers)),
        "roe": rng.uniform(0.04, 0.28, len(tickers)),
        "debt_to_equity": rng.uniform(0.0, 1.2, len(tickers)),
        "dividend_yield": rng.uniform(0.0, 0.07, len(tickers)),
        "company_scores_synthetic": True,
    })
    source_company_scores = "synthetic_company_scores"

# latest prices and history
latest_prices = company_scores[["ticker"]].copy()
latest_prices["price"] = rng.uniform(80, 220, len(latest_prices))
latest_prices["fx_rate"] = 1.0
latest_prices["date"] = pd.Timestamp(EXPERIMENT["end_date"])
dates = pd.bdate_range(EXPERIMENT["start_date"], EXPERIMENT["end_date"])
price_frames = []
for _, row in latest_prices.iterrows():
    ret = rng.normal(0.00025, 0.014, len(dates))
    close = float(row["price"]) / np.exp(np.cumsum(ret)[-1]) * np.exp(np.cumsum(ret))
    price_frames.append(pd.DataFrame({"date": dates, "ticker": row["ticker"], "close": close, "adj_close": close}))
price_panel = pd.concat(price_frames, ignore_index=True)
benchmark = pd.Series(100 * np.exp(np.cumsum(rng.normal(0.0002, 0.01, len(dates)))), index=dates, name="benchmark")

data_source_summary = pd.DataFrame([
    {"series": "transactions", "source": source_transactions, "rows": len(transactions)},
    {"series": "company_scores", "source": source_company_scores, "rows": len(company_scores)},
    {"series": "price_panel", "source": "synthetic_price_history_for_dashboard" if source_company_scores.startswith("synthetic") else "derived_price_history", "rows": len(price_panel)},
])
data_source_summary.to_csv(TABLES_DIR / "Table_1_data_source_summary.csv", index=False)
print("DATA SOURCE SUMMARY")
print(data_source_summary.to_string(index=False))


## 2. Cleaning & Alignment

Normalize transaction and holdings data, align tickers, currencies and latest company scores.

| Cell | What it does | Output |
|---|---|---|
| Main | Executes this section according to the SWS Portfolio Analysis Model standard | Saved tables/figures/dashboard artifacts |

$$CF_t=\pm q_t P_t FX_t - fees_t$$

Clean cash-flow signs are essential for dollar-weighted portfolio returns.


In [ ]:
# 2.1 Normalize holdings and align company scores
config = PortfolioConfig(base_currency=EXPERIMENT["base_currency"], benchmark_ticker=PORTFOLIO["benchmark"])
results = analyze_portfolio(transactions, latest_prices, company_scores, price_panel=price_panel, benchmark=benchmark, config=config)
holdings = results["holdings"]
company_scores_aligned = holdings[["ticker", "weight"]].merge(company_scores, on="ticker", how="left")
alignment = pd.DataFrame([{"metric":"holdings", "value":len(holdings), "N":len(holdings)}, {"metric":"score_coverage", "value":company_scores_aligned.dropna(subset=["sws_value_score"]).shape[0] / len(holdings), "N":len(holdings)}])
alignment.to_csv(TABLES_DIR / "Table_2_alignment.csv", index=False)
print(alignment.to_string(index=False))
display(holdings.head())


## 3. Feature Engineering

Build portfolio holdings, weights, snowflake aggregates and contributor features.

| Cell | What it does | Output |
|---|---|---|
| Main | Executes this section according to the SWS Portfolio Analysis Model standard | Saved tables/figures/dashboard artifacts |

$$Snowflake_p=\sum_i w_i Snowflake_i$$

The portfolio snowflake is a value-weighted view of individual company quality.


In [ ]:
# 3.1 Holdings, snowflake and contributors
snowflake = results["snowflake"]
contributors = results["contributors"]
metric_summary = results["metric_summary"]
for name, frame in [("holdings", holdings), ("snowflake", snowflake), ("contributors", contributors), ("metric_summary", metric_summary)]:
    frame.to_csv(TABLES_DIR / f"Table_3_{name}.csv", index=False)
print(snowflake.round(4).to_string(index=False))
print(contributors.round(4).to_string(index=False))


## 4. Targets & Labels

Define portfolio objectives and benchmark labels for return/risk diagnostics.

| Cell | What it does | Output |
|---|---|---|
| Main | Executes this section according to the SWS Portfolio Analysis Model standard | Saved tables/figures/dashboard artifacts |

$$ActiveReturn_t=R_{p,t}-R_{b,t}$$

Targets clarify whether the portfolio is being evaluated for absolute return, benchmark-relative return, or quality exposure.


In [ ]:
# 4.1 Portfolio targets and labels
returns = results["time_weighted_returns"]
overview = pd.DataFrame([results["overview"]])
if not returns.empty:
    returns["active_positive"] = (returns.get("active_return", 0) > 0).astype(float)
    returns["portfolio_drawdown"] = returns["portfolio_equity"] / returns["portfolio_equity"].cummax() - 1
returns.to_csv(TABLES_DIR / "Table_4_portfolio_returns_labels.csv", index=False)
overview.to_csv(TABLES_DIR / "Table_4_portfolio_overview.csv", index=False)
print(overview.round(4).to_string(index=False))


## 5. Descriptive Stats

Summarize holdings, concentration, sectors, countries and metric coverage.

| Cell | What it does | Output |
|---|---|---|
| Main | Executes this section according to the SWS Portfolio Analysis Model standard | Saved tables/figures/dashboard artifacts |

$$HHI=\sum_i w_i^2$$

Concentration and coverage determine how much confidence to place in weighted portfolio metrics.


In [ ]:
# 5.1 Descriptive stats
concentration = pd.DataFrame([{"metric":"HHI", "value":float((holdings["weight"]**2).sum()), "N":len(holdings)}, {"metric":"top_3_weight", "value":float(holdings.sort_values("weight", ascending=False).head(3)["weight"].sum()), "N":len(holdings)}, {"metric":"portfolio_value", "value":float(holdings["current_value_base"].sum()), "N":len(holdings)}])
concentration.to_csv(TABLES_DIR / "Table_5_descriptive_stats.csv", index=False)
print(concentration.round(4).to_string(index=False))


## 6. Exploratory / Event Study

Explore portfolio composition, snowflake axes and contributor decomposition.

| Cell | What it does | Output |
|---|---|---|
| Main | Executes this section according to the SWS Portfolio Analysis Model standard | Saved tables/figures/dashboard artifacts |

$$Contributor_{i,a}=Score_{i,a}\times w_i$$

Best/worst contributors reveal which holdings drive portfolio strengths and weaknesses.


In [ ]:
# 6.1 Composition and contributors charts
plt.figure(figsize=(10,5))
plot_holdings = holdings.sort_values("weight", ascending=False)
plt.bar(plot_holdings["ticker"], plot_holdings["weight"], color=COLORS["primary"])
plt.title("Portfolio weights", fontweight="bold")
plt.xlabel("Ticker")
plt.ylabel("Weight (fraction)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout(); plt.savefig(FIGURES_DIR / "Figure_1_portfolio_weights.png"); plt.show()

plt.figure(figsize=(10,5))
plt.bar(snowflake["axis"], snowflake["portfolio_score"], color=COLORS["accent"])
plt.title("Portfolio snowflake scores", fontweight="bold")
plt.xlabel("SWS axis")
plt.ylabel("Weighted score (0-6)")
plt.tight_layout(); plt.savefig(FIGURES_DIR / "Figure_2_snowflake_scores.png"); plt.show()


## 7. Single-Factor Diagnostics

Evaluate single portfolio metrics versus contribution and return diagnostics.

| Cell | What it does | Output |
|---|---|---|
| Main | Executes this section according to the SWS Portfolio Analysis Model standard | Saved tables/figures/dashboard artifacts |

$$\rho(metric_i, contribution_i)$$

Single-factor diagnostics identify whether returns are dominated by value, quality, income or concentration.


In [ ]:
# 7.1 Single-factor diagnostics
factor_rows = []
merged = holdings.merge(company_scores, on="ticker", how="left")
for metric in ["sws_value_score", "sws_future_score", "sws_past_score", "sws_health_score", "sws_income_score", "pe_ratio", "pb_ratio", "roe", "dividend_yield"]:
    if metric in merged.columns:
        valid = merged[[metric, "weight"]].dropna()
        factor_rows.append({"metric": metric, "corr_with_weight": valid[metric].corr(valid["weight"], method="spearman") if len(valid) >= 3 else np.nan, "weighted_average": (valid[metric] * valid["weight"]).sum() / valid["weight"].sum() if not valid.empty else np.nan, "N": len(valid)})
single_factor = pd.DataFrame(factor_rows)
single_factor.to_csv(TABLES_DIR / "Table_7_single_factor.csv", index=False)
print(single_factor.round(4).to_string(index=False))


## 8. Statistical Models (regressions / econometrics)

Run simple portfolio attribution regressions where return history is available.

| Cell | What it does | Output |
|---|---|---|
| Main | Executes this section according to the SWS Portfolio Analysis Model standard | Saved tables/figures/dashboard artifacts |

$$R_p=\alpha+\beta R_b+\epsilon$$

Benchmark beta and alpha provide an interpretable return attribution baseline.


In [ ]:
# 8.1 Benchmark attribution regression
regression_rows = []
if not returns.empty and {"portfolio_return", "benchmark_return"} <= set(returns.columns):
    valid = returns[["portfolio_return", "benchmark_return"]].dropna()
    if len(valid) >= 30:
        x = valid[["benchmark_return"]].to_numpy()
        y = valid["portfolio_return"].to_numpy()
        beta = np.cov(x[:,0], y)[0,1] / np.var(x[:,0]) if np.var(x[:,0]) else np.nan
        alpha = y.mean() - beta * x[:,0].mean() if np.isfinite(beta) else np.nan
        regression_rows.append({"model":"portfolio_vs_benchmark", "alpha_daily":alpha, "beta":beta, "N":len(valid)})
regression = pd.DataFrame(regression_rows)
regression.to_csv(TABLES_DIR / "Table_8_attribution_regression.csv", index=False)
print(regression.round(4).to_string(index=False))


## 9. ML Walk-Forward

Run a compact walk-forward diagnostic on holding-level features when enough history exists.

| Cell | What it does | Output |
|---|---|---|
| Main | Executes this section according to the SWS Portfolio Analysis Model standard | Saved tables/figures/dashboard artifacts |

$$train<t,\ test=t$$

The portfolio notebook keeps ML secondary; it is a diagnostic, not a substitute for transparent allocation review.


In [ ]:
# 9.1 Compact walk-forward diagnostic placeholder
ml_metrics = pd.DataFrame([{"model":"not_applicable_portfolio_level", "reason":"Portfolio model is aggregation-first; ML belongs in company-level pipeline", "N":len(holdings)}])
ml_metrics.to_csv(TABLES_DIR / "Table_9_ml_walk_forward.csv", index=False)
print(ml_metrics.to_string(index=False))


## 10. Feature Ablation

Ablate portfolio scoring axes to show their contribution to ranking and allocation.

| Cell | What it does | Output |
|---|---|---|
| Main | Executes this section according to the SWS Portfolio Analysis Model standard | Saved tables/figures/dashboard artifacts |

$$\Delta_a=Score_{all}-Score_{-a}$$

Axis ablation reveals whether the portfolio depends on one fragile score dimension.


In [ ]:
# 10.1 Axis ablation
base_score = snowflake["portfolio_score"].mean()
ablation = snowflake.copy()
ablation["score_without_axis"] = [(snowflake.loc[snowflake["axis"] != axis, "portfolio_score"].mean()) for axis in snowflake["axis"]]
ablation["delta_vs_all"] = ablation["score_without_axis"] - base_score
ablation.to_csv(TABLES_DIR / "Table_10_ablation.csv", index=False)
plt.figure(figsize=(10,5)); plt.barh(ablation["axis"], ablation["delta_vs_all"], color=COLORS["purple"]); plt.axvline(0, color="black", linestyle="--", lw=0.8); plt.title("Snowflake axis ablation", fontweight="bold"); plt.xlabel("Delta score"); plt.ylabel("Removed axis"); plt.tight_layout(); plt.savefig(FIGURES_DIR / "Figure_3_axis_ablation.png"); plt.show()
print(ablation.round(4).to_string(index=False))


## 11. Backtest / Strategy Evaluation

Compute money-weighted, time-weighted and benchmark-relative returns.

| Cell | What it does | Output |
|---|---|---|
| Main | Executes this section according to the SWS Portfolio Analysis Model standard | Saved tables/figures/dashboard artifacts |

$$CAGR=(1+TR)^{1/AYI}-1$$

Dollar-weighted return is appropriate when the investor controls cash-flow timing.


In [ ]:
# 11.1 Return evaluation
return_summary = pd.DataFrame([results["overview"]])
return_summary.to_csv(TABLES_DIR / "Table_11_strategy_evaluation.csv", index=False)
plt.figure(figsize=(10,5))
if not returns.empty:
    plt.plot(returns["date"], returns["portfolio_equity"], label="Portfolio", color=COLORS["primary"])
    if "benchmark_equity" in returns.columns:
        plt.plot(returns["date"], returns["benchmark_equity"], label="Benchmark", color=COLORS["accent"])
plt.axhline(1, color="black", linestyle="--", lw=0.8)
plt.title("Portfolio vs benchmark equity", fontweight="bold")
plt.xlabel("Date")
plt.ylabel("Growth of 1")
plt.legend(); plt.tight_layout(); plt.savefig(FIGURES_DIR / "Figure_4_portfolio_vs_benchmark.png"); plt.show()
print(return_summary.round(4).to_string(index=False))


## 12. Interpretability

Explain portfolio strengths/weaknesses using contributors and metric exposures.

| Cell | What it does | Output |
|---|---|---|
| Main | Executes this section according to the SWS Portfolio Analysis Model standard | Saved tables/figures/dashboard artifacts |

$$Impact_i=w_i\times metric_i$$

Interpretability maps portfolio-level conclusions back to holdings.


In [ ]:
# 12.1 Interpretability via contributors
interpretability = contributors.sort_values("contribution", ascending=False)
interpretability.to_csv(TABLES_DIR / "Table_12_interpretability.csv", index=False)
print(interpretability.round(4).to_string(index=False))


## 13. Robustness Checks

Stress concentration, costs, benchmark choice and missing fair-value coverage.

| Cell | What it does | Output |
|---|---|---|
| Main | Executes this section according to the SWS Portfolio Analysis Model standard | Saved tables/figures/dashboard artifacts |

$$Stress=Metric(\theta+\Delta)-Metric(\theta)$$

Robustness checks make explicit which conclusions depend on assumptions.


In [ ]:
# 13.1 Robustness checks: concentration, value coverage, cost sensitivity
robustness = []
robustness.append({"test":"intrinsic_value_coverage", "metric":results["overview"].get("coverage_weight", np.nan), "N":len(holdings)})
robustness.append({"test":"top_1_weight", "metric":holdings["weight"].max(), "N":len(holdings)})
for haircut in [0.0, 0.1, 0.2]:
    stressed = holdings["current_value_base"].sum() * (1 - haircut)
    robustness.append({"test":"portfolio_value_haircut", "variant":haircut, "metric":stressed, "N":len(holdings)})
robustness = pd.DataFrame(robustness)
robustness.to_csv(TABLES_DIR / "Table_13_robustness.csv", index=False)
print(robustness.round(4).to_string(index=False))


## 14. Final Summary

Export CSV, Excel workbook and interactive HTML dashboard, then print final artifact inventory.

| Cell | What it does | Output |
|---|---|---|
| Main | Executes this section according to the SWS Portfolio Analysis Model standard | Saved tables/figures/dashboard artifacts |

$$Report=\{CSV,XLSX,HTML,Figures\}$$

The final deliverable is a shareable research artifact rather than a raw notebook dump.


In [ ]:
# 14.1 Export CSV, Excel and dashboard
exported = export_portfolio_analysis(results, TABLES_DIR)
excel_path = TABLES_DIR / "portfolio_analysis_report.xlsx"
excel_available = importlib.util.find_spec("xlsxwriter") is not None or importlib.util.find_spec("openpyxl") is not None
if excel_available:
    excel_engine = "xlsxwriter" if importlib.util.find_spec("xlsxwriter") is not None else "openpyxl"
    with pd.ExcelWriter(excel_path, engine=excel_engine) as writer:
        for name, obj in results.items():
            if isinstance(obj, pd.DataFrame):
                obj.to_excel(writer, sheet_name=name[:31], index=False)
            else:
                pd.DataFrame([obj]).to_excel(writer, sheet_name=name[:31], index=False)
        concentration.to_excel(writer, sheet_name="descriptive", index=False)
        robustness.to_excel(writer, sheet_name="robustness", index=False)
else:
    excel_path = TABLES_DIR / "portfolio_analysis_report_excel_unavailable.txt"
    excel_path.write_text("Install xlsxwriter or openpyxl to export the Excel workbook. CSV and HTML artifacts were still generated.", encoding="utf-8")
    logger.warning("Excel export skipped because neither xlsxwriter nor openpyxl is installed")

if PLOTLY_AVAILABLE:
    fig_weights = px.bar(holdings.sort_values("weight", ascending=False), x="ticker", y="weight", title="Portfolio weights", template="plotly_white")
    fig_snowflake = px.bar(snowflake, x="axis", y="portfolio_score", title="Portfolio snowflake", template="plotly_white")
    charts_html = fig_weights.to_html(full_html=False, include_plotlyjs="cdn") + fig_snowflake.to_html(full_html=False, include_plotlyjs=False)
else:
    charts_html = "<p>Static figures saved in output/figures.</p>"

def table_html(frame):
    return frame.to_html(index=False, float_format=lambda x: f"{x:,.4f}")
html = f"""
<html><head><title>SWS Portfolio Analysis Dashboard</title>
<style>
body {{font-family: Arial, sans-serif; background:{COLORS['bg']}; margin:24px; color:#1f2933;}}
.header {{background:{COLORS['primary']}; color:white; padding:22px; border-radius:14px;}}
.card {{display:inline-block; background:white; margin:10px; padding:15px; border-left:5px solid {COLORS['accent']}; border-radius:10px; min-width:180px;}}
.section {{background:white; margin:18px 0; padding:18px; border-radius:12px;}}
th {{background:{COLORS['primary']}; color:white; padding:6px;}} td {{padding:6px; border-bottom:1px solid #ddd;}}
</style></head><body>
<div class='header'><h1>SWS Portfolio Analysis Dashboard</h1><p>Money-weighted returns, portfolio snowflake, contributors, intrinsic value and benchmark diagnostics.</p></div>
<div class='card'><b>Portfolio value</b><br>{results['overview'].get('portfolio_value_base', np.nan):,.2f}</div>
<div class='card'><b>Total return</b><br>{results['overview'].get('total_return', np.nan):.2%}</div>
<div class='card'><b>Annualized return</b><br>{results['overview'].get('annualized_return', np.nan):.2%}</div>
<div class='card'><b>Intrinsic upside</b><br>{results['overview'].get('upside_to_intrinsic', np.nan):.2%}</div>
<div class='section'><h2>Executive Overview</h2>{table_html(return_summary)}</div>
<div class='section'><h2>Holdings</h2>{table_html(holdings)}</div>
<div class='section'><h2>Portfolio Snowflake</h2>{table_html(snowflake)}</div>
<div class='section'><h2>Best/Worst Contributors</h2>{table_html(contributors)}</div>
<div class='section'><h2>Robustness</h2>{table_html(robustness)}</div>
<div class='section'><h2>Interactive Charts</h2>{charts_html}</div>
</body></html>
"""
dashboard_path = DASHBOARD_DIR / "portfolio_analysis_dashboard.html"
dashboard_path.write_text(html, encoding="utf-8")
manifest = {"excel": str(excel_path), "dashboard": str(dashboard_path), "exported_csv": exported}
(TABLES_DIR / "portfolio_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("Excel artifact:", excel_path)
print("Dashboard exported:", dashboard_path)


In [ ]:
import os
from pathlib import Path
tables  = sorted(Path("output/tables").glob("*.csv"))
figures = sorted(Path("output/figures").glob("*.png"))
print(f"✅ EXPERIMENT COMPLETE")
print(f"   Tables : {len(tables)}")
print(f"   Figures: {len(figures)}")
for f in tables:  print(f"   📋 {f.name}")
for f in figures: print(f"   📊 {f.name}")
